In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd drive/MyDrive/

In [ ]:
!rm -rf FPL_forecast
!git clone https://github.com/bragehs/FPL_forecast.git

In [ ]:
%cd FPL_forecast/predictor/

In [ ]:
file_path = '/content/drive/MyDrive/colab_fpl'
file_path

In [ ]:
import os
import torch
from training import train_model, hyperparameter_tuning
from model import AdvancedLSTM

In [ ]:
X_train = torch.load(file_path + "/X_train.pt", weights_only=True)
y_train = torch.load(file_path + "/y_train.pt", weights_only=True)
X_val = torch.load(file_path + "/X_val.pt", weights_only=True)
y_val = torch.load(file_path + "/y_val.pt", weights_only=True)

print(f"Train sequences: {X_train.shape}, Targets: {y_train.shape}")

Input dimension: 48


In [3]:
# Hyperparameter tuning
best_params = hyperparameter_tuning(X_train, y_train, X_val, y_val, epochs=30, n_trials=25, transform=False)

# Full training with best hyperparameters
print("\nTraining final model with best hyperparameters...")
adv_model = AdvancedLSTM(input_dim=X_train.shape[-1], hidden_dim=best_params['hidden_dim'],
                            output_dim=1, num_layers=best_params['num_layers'],
                            dropout=best_params['dropout'], num_fc_layers=best_params['num_fc_layers'])


train_model(
    adv_model,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    learning_rate=best_params['learning_rate'],
    weight_decay=best_params['weight_decay'],
    batch_size=best_params['batch_size'],
    epochs=100,  # Full training
    verbose=2,
    transform=False,
)

Running random search with 25 trials...

Trial 1/25
Params: {'learning_rate': 0.0005, 'hidden_dim': 64, 'weight_decay': 0.001, 'num_layers': 4, 'dropout': 0.4, 'num_fc_layers': 1, 'batch_size': 256}


KeyboardInterrupt: 